# Scraping Data Ulasan Play Store (Indonesia)

Notebook ini mengambil data ulasan aplikasi Android secara mandiri dari internet menggunakan `google-play-scraper`.

Target:
- Minimal 10.000 sampel
- Data disimpan ke CSV untuk dipakai pada notebook analisis sentimen

Sumber data:
- Google Play Store (beberapa aplikasi populer di Indonesia)

In [1]:
%pip install -q google-play-scraper pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 811.5 kB/s eta 0:00:00


In [2]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import Sort, reviews

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Daftar aplikasi populer Indonesia (lintas kategori)
APPS = {
    "com.gojek.app": "Gojek",
    "com.tokopedia.tkpd": "Tokopedia",
    "com.shopee.id": "Shopee",
    "com.traveloka.android": "Traveloka",
    "id.dana": "DANA",
}

LANG = "id"
COUNTRY = "id"
TARGET_PER_APP = 2200  # 5 aplikasi x 2200 ~= 11.000 ulasan
MAX_BATCH = 200

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "reviews_playstore_id.csv"

print(f"Target total sampel: ~{TARGET_PER_APP * len(APPS):,}".replace(",", "."))

Target total sampel: ~11.000


In [3]:
def scrape_reviews_for_app(app_id: str, app_name: str, target_count: int = 2000):
    """Scrape review Play Store per aplikasi sampai target terpenuhi atau data habis."""
    all_rows = []
    token = None

    while len(all_rows) < target_count:
        batch_count = min(MAX_BATCH, target_count - len(all_rows))
        result, token = reviews(
            app_id,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=batch_count,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append(
                {
                    "app_id": app_id,
                    "app_name": app_name,
                    "review_id": r.get("reviewId"),
                    "user_name": r.get("userName"),
                    "score": r.get("score"),
                    "content": r.get("content"),
                    "thumbs_up": r.get("thumbsUpCount"),
                    "at": r.get("at"),
                    "app_version": r.get("appVersion"),
                }
            )

        if token is None:
            break

    return all_rows

In [4]:
all_reviews = []

for app_id, app_name in tqdm(APPS.items(), desc="Scraping aplikasi"):
    rows = scrape_reviews_for_app(app_id, app_name, target_count=TARGET_PER_APP)
    all_reviews.extend(rows)
    print(f"{app_name:10s} -> {len(rows):,} ulasan".replace(",", "."))

raw_df = pd.DataFrame(all_reviews)
print(f"\nTotal ulasan terkumpul (raw): {len(raw_df):,}".replace(",", "."))
raw_df.head()

Scraping aplikasi:   0%|          | 0/5 [00:00<?, ?it/s]

Gojek      -> 2.200 ulasan
Tokopedia  -> 2.200 ulasan
Shopee     -> 2.200 ulasan
Traveloka  -> 2.200 ulasan
DANA       -> 2.200 ulasan

Total ulasan terkumpul (raw): 11.000


,app_id,app_name,review_id,user_name,score,content,thumbs_up,at,app_version
0,com.gojek.app,Gojek,0321bdf5-9df4-4527-a0d4-db0b7692d327,Rozali Yogi Hadiid Prananta,5,gooddd,0,2026-04-25 18:26:33,5.56.2
1,com.gojek.app,Gojek,c1227ba5-437f-4fe5-9167-a3b54e5ea7b6,Ajeng Kartini Sulastrina,1,mahal ongkir food tolong diperkecil lagi.hai guys kita ramaikan city plasa kuta bumi tangerang tiap senin sampai jum...,0,2026-04-25 17:43:31,None
2,com.gojek.app,Gojek,d944a379-6044-4590-8aef-14c095aed2c0,Bagus Nuryadin,1,sistem dibenerin bos,0,2026-04-25 17:12:39,5.57.2
3,com.gojek.app,Gojek,9f449a40-6dec-422b-bc48-8b3f8d7ea809,Dian maulana,1,jelek banget,0,2026-04-25 16:53:21,5.54.1
4,com.gojek.app,Gojek,854740cd-545d-41ce-ad75-061e2852c7fb,Maha Rani,5,mantapp,0,2026-04-25 16:44:26,5.5.1


In [5]:
df = raw_df.copy()

# Pembersihan dasar
before = len(df)
df = df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
df = df.dropna(subset=["content", "score"])
df["content"] = df["content"].astype(str).str.strip()
df = df[df["content"].str.len() > 3].reset_index(drop=True)

df["at"] = pd.to_datetime(df["at"], errors="coerce")

print(f"Sebelum cleaning : {before:,}".replace(",", "."))
print(f"Setelah cleaning : {len(df):,}".replace(",", "."))
print("Distribusi skor:")
display(df["score"].value_counts().sort_index())

# Simpan ke CSV
ordered_cols = [
    "review_id", "app_id", "app_name", "user_name", "score", "content", "thumbs_up", "at", "app_version"
]
existing_cols = [c for c in ordered_cols if c in df.columns]
df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nFile CSV tersimpan di: {OUTPUT_CSV.resolve()}")
print(f"Jumlah sampel akhir: {len(df):,}".replace(",", "."))

Sebelum cleaning : 11.000
Setelah cleaning : 10.412
Distribusi skor:


,count
score,
1,2844
2,407
3,465
4,595
5,6101



File CSV tersimpan di: /content/data/reviews_playstore_id.csv
Jumlah sampel akhir: 10.412


In [6]:
df.sample(5, random_state=42)[["app_name", "score", "content"]]

,app_name,score,content
7935,Traveloka,1,karenakan tpaylater nya di bekukan setelah sekian lama tak kunjung aktif juga.. dengan ini saya berenti memakai trav...
8779,DANA,2,"dana cicil tidak bisah di aktivasi, di cek lagi udah hilang"
8514,DANA,5,baik
3781,Tokopedia,5,bagus dan reguler sip
2517,Tokopedia,5,sangat membatu
